In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import (
    BitsAndBytesConfig,
    Gemma4ForConditionalGeneration,
    Gemma4Processor,
    set_seed,
)
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer

In [ ]:
MODEL_BASE = "google/gemma-4-E2B-it"
MODEL_OUTPUT_DIR = "gemma-4-E2B-it-finetune"

set_seed(42)

In [ ]:
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
dataset = dataset.shuffle()

In [ ]:
processor = Gemma4Processor.from_pretrained(MODEL_BASE)

In [ ]:
model = Gemma4ForConditionalGeneration.from_pretrained(
    MODEL_BASE,
    vision_config=None,
    audio_config=None,
    dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_storage=torch.bfloat16,
    ),
)

In [ ]:
split = dataset.train_test_split(test_size=0.1)

args = SFTConfig(
    output_dir=MODEL_OUTPUT_DIR,  # directory to save and repository id
    max_length=512,  # max length for model and packing of the dataset
    num_train_epochs=3,  # number of training epochs
    per_device_train_batch_size=1,  # batch size per device during training
    optim="adamw_torch_fused",  # use fused adamw optimizer
    logging_steps=10,  # log every 10 steps
    save_strategy="epoch",  # save checkpoint every epoch
    eval_strategy="epoch",  # evaluate checkpoint every epoch
    learning_rate=5e-5,  # learning rate
    fp16=True if model.dtype == torch.float16 else False,  # use float16 precision
    bf16=True if model.dtype == torch.bfloat16 else False,  # use bfloat16 precision
    max_grad_norm=0.3,  # max gradient norm based on QLoRA paper
    lr_scheduler_type="constant",  # use constant learning rate scheduler
    # push_to_hub=True,  # push model to hub
    # report_to="tensorboard",  # report metrics to tensorboard
    # report_to="trackio",
    dataset_kwargs={
        "add_special_tokens": False,  # Template with special tokens
        "append_concat_token": True,  # Add EOS token as separator token between examples
    },
)

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    # make sure to save the lm_head and embed_tokens as you train the special tokens
    modules_to_save=["lm_head", "embed_tokens"],
    ensure_weight_tying=True,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    peft_config=peft_config,
    processing_class=processor,
)

In [ ]:
trainer.train()
trainer.save_model()